# Fractal Gap — Hierarchy of Absence Notebook
## Orbit / Seam / GF(2) Jacobian / Universal Seed / Rotation-Span Workbench

This notebook is the computational companion to the **Fractal Gap and Hierarchy of Absence** paper.

It is organized in three explicit layers:

1. **Hard die layer** — exact SHA-256 round algebra, seam definitions, carry-save decomposition, and transport identities.
2. **Measured geometry layer** — GF(2) Jacobian construction, rank/null-space extraction, parity-check derivation, and rotation-span experiments.
3. **Interpretive hierarchy layer** — the paper’s three-resolution map:
   - **Orbit**: macro gap,
   - **Seam**: carry-residual operational layer,
   - **Jacobian**: foundational blind subspace.

The notebook is self-contained. It does **not** require external `.py` files.

The working discipline is strict:

- prove what can be proved directly,
- measure what depends on implementation choice,
- label the larger lift as a structural map rather than pretending every claim is already closed.

The spine is:

$$
\boxed{
\text{Orbit} \to \text{Seam} \to \text{GF(2) Jacobian} \to \text{Universal Seed } B \to \text{Parity Recovery}
}
$$

and the paper’s central seed is

$$
\boxed{
B = \Sigma_0(a)\land \operatorname{Maj}(a,b,c).
}
$$


In [ ]:

import math
import random
from dataclasses import dataclass
from typing import List, Tuple, Dict

import numpy as np
import matplotlib.pyplot as plt

MASK32 = 0xFFFFFFFF

# SHA-256 IV and round constants
H0 = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
]

K = [
    0x428A2F98, 0x71374491, 0xB5C0FBCF, 0xE9B5DBA5, 0x3956C25B, 0x59F111F1, 0x923F82A4, 0xAB1C5ED5,
    0xD807AA98, 0x12835B01, 0x243185BE, 0x550C7DC3, 0x72BE5D74, 0x80DEB1FE, 0x9BDC06A7, 0xC19BF174,
    0xE49B69C1, 0xEFBE4786, 0x0FC19DC6, 0x240CA1CC, 0x2DE92C6F, 0x4A7484AA, 0x5CB0A9DC, 0x76F988DA,
    0x983E5152, 0xA831C66D, 0xB00327C8, 0xBF597FC7, 0xC6E00BF3, 0xD5A79147, 0x06CA6351, 0x14292967,
    0x27B70A85, 0x2E1B2138, 0x4D2C6DFC, 0x53380D13, 0x650A7354, 0x766A0ABB, 0x81C2C92E, 0x92722C85,
    0xA2BFE8A1, 0xA81A664B, 0xC24B8B70, 0xC76C51A3, 0xD192E819, 0xD6990624, 0xF40E3585, 0x106AA070,
    0x19A4C116, 0x1E376C08, 0x2748774C, 0x34B0BCB5, 0x391C0CB3, 0x4ED8AA4A, 0x5B9CCA4F, 0x682E6FF3,
    0x748F82EE, 0x78A5636F, 0x84C87814, 0x8CC70208, 0x90BEFFFA, 0xA4506CEB, 0xBEF9A3F7, 0xC67178F2,
]

def rotr(x: int, n: int) -> int:
    x &= MASK32
    return ((x >> n) | ((x << (32 - n)) & MASK32)) & MASK32

def Sigma0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def Sigma1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def sigma0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ ((x >> 3) & MASK32)

def sigma1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ ((x >> 10) & MASK32)

def Ch(e: int, f: int, g: int) -> int:
    return ((e & f) ^ ((~e) & g)) & MASK32

def Maj(a: int, b: int, c: int) -> int:
    return ((a & b) ^ (a & c) ^ (b & c)) & MASK32

@dataclass
class RoundObs:
    r: int
    state: List[int]
    t1: int
    t2: int
    seam_xor: int
    seam_full: int
    carry_residual: int
    seed_B: int

def round_terms(state: List[int], W_r: int, r: int) -> Tuple[int, int]:
    a, b, c, d, e, f, g, h = state
    t1 = (h + Sigma1(e) + Ch(e, f, g) + K[r] + W_r) & MASK32
    t2 = (Sigma0(a) + Maj(a, b, c)) & MASK32
    return t1, t2

def round_step(state: List[int], W_r: int, r: int) -> List[int]:
    a, b, c, d, e, f, g, h = state
    t1, t2 = round_terms(state, W_r, r)
    new_a = (t1 + t2) & MASK32
    new_e = (d + t1) & MASK32
    return [new_a, a, b, c, new_e, e, f, g]

def seam_xor_word(state: List[int]) -> int:
    a, b, c, d, e, f, g, h = state
    return (Sigma0(a) ^ Maj(a, b, c) ^ d) & MASK32

def seam_full_word(state: List[int]) -> int:
    a, b, c, d, e, f, g, h = state
    return (Sigma0(a) + Maj(a, b, c) + d) & MASK32

def universal_seed_B(state: List[int]) -> int:
    a, b, c, d, e, f, g, h = state
    return (Sigma0(a) & Maj(a, b, c)) & MASK32

def observe_orbit(Ws: List[int], rounds: int = 6, state0: List[int] = None) -> List[RoundObs]:
    state = (state0 or H0).copy()
    obs = []
    for r in range(rounds):
        t1, t2 = round_terms(state, Ws[r], r)
        sx = seam_xor_word(state)
        sf = seam_full_word(state)
        B = universal_seed_B(state)
        obs.append(RoundObs(
            r=r,
            state=state.copy(),
            t1=t1,
            t2=t2,
            seam_xor=sx,
            seam_full=sf,
            carry_residual=sf ^ sx,
            seed_B=B
        ))
        state = round_step(state, Ws[r], r)
    return obs


## 1. Hard anchors

These are the exact die anchors we already know how to verify:

1. Ground fold at round 0:
   $$
   T2_0^{(0)} = 0x08909ae5
   $$
2. First-step message displacement:
   $$
   T1_0 - T1_0^{(0)} = W_0
   $$
3. Seam differential:
   $$
   a_{r+1} - e_{r+1} \equiv T2_r - d_r \pmod{2^{32}}
   $$


In [ ]:

# Ground witness
t1_0, t2_0 = round_terms(H0.copy(), 0, 0)
print(f"T2_0^(0) = 0x{t2_0:08x}")
assert t2_0 == 0x08909AE5

# First-step displacement
for W0 in [0, 1, 0x80000000, 0x12345678, 0xFFFFFFFF]:
    base = round_step(H0.copy(), 0, 0)
    live = round_step(H0.copy(), W0, 0)
    # Compare T1 directly
    t1_base, _ = round_terms(H0.copy(), 0, 0)
    t1_live, _ = round_terms(H0.copy(), W0, 0)
    assert ((t1_live - t1_base) & MASK32) == (W0 & MASK32)
    assert ((live[0] - base[0]) & MASK32) == (W0 & MASK32)
    assert ((live[4] - base[4]) & MASK32) == (W0 & MASK32)

# Seam differential
rng = random.Random(20260403)
for _ in range(4000):
    state = [rng.getrandbits(32) for _ in range(8)]
    W_r = rng.getrandbits(32)
    r = rng.randrange(64)
    t1, t2 = round_terms(state, W_r, r)
    new_state = round_step(state, W_r, r)
    lhs = (new_state[0] - new_state[4]) & MASK32
    rhs = (t2 - state[3]) & MASK32
    assert lhs == rhs

print("Verified:")
print("  • T2_0^(0) ground witness")
print("  • T1_0 - T1_0^(0) = W0")
print("  • a[r+1] - e[r+1] = T2[r] - d[r]  (mod 2^32)")


## 2. Orbit and seam table

This cell gives the first six rounds in the chosen basis:

- `seam_xor` = XOR channel,
- `carry_residual` = `seam_full XOR seam_xor`,
- `seed_B` = `Sigma0(a) AND Maj(a,b,c)`.

**Important:** exact seam values depend on the chosen implementation basis and whether the state is observed pre-update or post-update. The notebook computes the basis explicitly rather than pretending there is only one canonical representation.


In [ ]:

obs = observe_orbit([0, 0, 0, 0, 0, 0], rounds=6)

print("r  seam_xor    seam_full   carry_res   seed_B")
for row in obs:
    print(
        f"{row.r:<2d}  "
        f"{row.seam_xor:08x}  "
        f"{row.seam_full:08x}  "
        f"{row.carry_residual:08x}  "
        f"{row.seed_B:08x}"
    )

print("\nAverage carry-residual Hamming weight:",
      sum(int(x.carry_residual).bit_count() for x in obs) / len(obs))


## 3. GF(2) utilities

The hierarchy-of-absence section of the paper ultimately lives or dies on the rank/null-space geometry of a chosen Jacobian.

We therefore build:

- GF(2) row-reduction,
- rank,
- null-space basis,
- left-nullspace / parity-check extraction.

This lets the notebook work directly in the same language as the paper, without hiding behind ordinary floating-point linear algebra.


In [ ]:

def gf2_rref(A: np.ndarray):
    A = (A.copy() & 1).astype(np.uint8)
    m, n = A.shape
    pivots = []
    row = 0
    for col in range(n):
        pivot = None
        for r in range(row, m):
            if A[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        if pivot != row:
            A[[row, pivot]] = A[[pivot, row]]
        for r in range(m):
            if r != row and A[r, col]:
                A[r] ^= A[row]
        pivots.append(col)
        row += 1
        if row == m:
            break
    return A, pivots

def gf2_rank(A: np.ndarray) -> int:
    _, pivots = gf2_rref(A)
    return len(pivots)

def gf2_nullspace_basis(A: np.ndarray) -> np.ndarray:
    R, pivots = gf2_rref(A)
    m, n = R.shape
    pivot_set = set(pivots)
    free_cols = [j for j in range(n) if j not in pivot_set]
    basis = []
    for fc in free_cols:
        v = np.zeros(n, dtype=np.uint8)
        v[fc] = 1
        # back-substitute pivot vars from RREF
        for i, pc in enumerate(pivots):
            if R[i, fc]:
                v[pc] = 1
        basis.append(v)
    if not basis:
        return np.zeros((n, 0), dtype=np.uint8)
    return np.stack(basis, axis=1)

def gf2_left_nullspace_basis(A: np.ndarray) -> np.ndarray:
    return gf2_nullspace_basis(A.T)

def bits_of_word(x: int, width: int = 32) -> np.ndarray:
    return np.array([(x >> b) & 1 for b in range(width)], dtype=np.uint8)

def flatten_word_list(words: List[int], width: int = 32) -> np.ndarray:
    return np.concatenate([bits_of_word(w, width) for w in words], axis=0)


## 4. Build a six-round / six-word seam Jacobian

This is the critical measurement section.

We choose an implementation-specific map

$$
W[0..5] \mapsto \text{observable}
$$

and compute its binary Jacobian by single-bit toggles around a base point.

By default, the base point is all-zero message words, and the observable is the 192-bit concatenation of `seam_xor` over rounds 0–5.

You can later swap in:
- post-update seam,
- different state anchor,
- augmented outputs,
- or a paper-specific seam basis.


In [ ]:

def seam_observable_from_W6(W6: List[int], basis: str = "pre_update_xor") -> np.ndarray:
    obs = observe_orbit(W6, rounds=6)
    if basis == "pre_update_xor":
        words = [row.seam_xor for row in obs]
    elif basis == "pre_update_full":
        words = [row.seam_full for row in obs]
    elif basis == "carry_residual":
        words = [row.carry_residual for row in obs]
    elif basis == "seed_B":
        words = [row.seed_B for row in obs]
    elif basis == "seam_plus_d_track":
        words = []
        for row in obs:
            words.append(row.seam_xor)
            words.append(row.state[3])  # d-register track
    elif basis == "seam_plus_a_track":
        words = []
        for row in obs:
            words.append(row.seam_xor)
            words.append(row.state[0])  # a-track
    elif basis == "seam_plus_e_track":
        words = []
        for row in obs:
            words.append(row.seam_xor)
            words.append(row.state[4])  # e-track
    else:
        raise ValueError(f"Unknown basis: {basis}")
    return flatten_word_list(words)

def build_jacobian_W6(basis: str = "pre_update_xor") -> np.ndarray:
    base_W = [0] * 6
    base_y = seam_observable_from_W6(base_W, basis=basis)
    out_dim = len(base_y)
    J = np.zeros((out_dim, 192), dtype=np.uint8)
    col = 0
    for j in range(6):
        for b in range(32):
            W = [0] * 6
            W[j] = (1 << b)
            y = seam_observable_from_W6(W, basis=basis)
            J[:, col] = base_y ^ y
            col += 1
    return J

J_seam = build_jacobian_W6("pre_update_xor")
rank_seam = gf2_rank(J_seam)
null_seam = gf2_nullspace_basis(J_seam)
left_null_seam = gf2_left_nullspace_basis(J_seam)

print("J_seam shape =", J_seam.shape)
print("rank(J_seam) =", rank_seam)
print("nullity(J_seam) =", J_seam.shape[1] - rank_seam)
print("left-nullity(J_seam) =", J_seam.shape[0] - rank_seam)


## 5. Parity-check matrix and null-space basis

If the seam map is rank-deficient, the null-space basis \(N\) and parity-check basis \(H\) become the mechanical heart of the inverse.

This cell extracts both.

- \(N\) spans the input directions invisible to the chosen observable.
- \(H\) spans the output constraints: valid output vectors satisfy \(Hy = 0\) in the linearized model.


In [ ]:

N = null_seam
H = left_null_seam.T  # rows = parity checks

print("N shape (input nullspace basis) =", N.shape)
print("H shape (parity-check basis)    =", H.shape)

# quick sanity check: H annihilates every column-space vector Jx
for trial in range(10):
    x = np.random.randint(0, 2, size=J_seam.shape[1], dtype=np.uint8)
    y = (J_seam @ x) & 1
    assert np.all(((H @ y) & 1) == 0)

print("Verified: H annihilates the linearized seam image.")


## 6. Visualize the Jacobian

This gives you a direct picture of the measured topology for the chosen seam basis.

A block-lower-triangular pattern means the causal cone is behaving as expected:
inputs from \(W[j]\) only begin influencing later rounds.


In [ ]:

plt.figure(figsize=(10, 8))
plt.imshow(J_seam, aspect='auto', cmap='binary', interpolation='nearest')
plt.title(f"GF(2) Jacobian heatmap — seam basis — rank {rank_seam}/{J_seam.shape[1]}")
plt.xlabel("input bit index (6 words × 32 bits)")
plt.ylabel("output bit index (6 rounds × 32 bits)")
plt.tight_layout()
plt.show()


## 7. Rotation-vector experiments

The paper hypothesis says the blind subspace may be architected by the rotation sets:

$$
\{2,13,22\}, \qquad \{6,11,25\}.
$$

This section builds simple binary support vectors for those rotation masks and compares them to:

- the right null-space basis of the seam map,
- the left null-space / parity-check basis of the seam image.

This does **not** yet prove span. It gives a measurable first probe.


In [ ]:

def rotation_support_vector(rotset: List[int], width: int = 32) -> np.ndarray:
    """Binary support vector marking all positions hit by the given rotation set from bit 0."""
    v = np.zeros(width, dtype=np.uint8)
    for r in rotset:
        v[(0 - r) % width] ^= 1  # use support-mark convention
    return v

rotA = rotation_support_vector([2, 13, 22], 32)
rotB = rotation_support_vector([6, 11, 25], 32)

print("rotA support weight =", int(rotA.sum()), rotA.tolist())
print("rotB support weight =", int(rotB.sum()), rotB.tolist())

# project 32-bit prototypes into 192-bit repeated round pattern
rotA_192 = np.tile(rotA, 6)
rotB_192 = np.tile(rotB, 6)

def overlap_score(v: np.ndarray, basis: np.ndarray) -> List[int]:
    if basis.size == 0:
        return []
    return [int((v & basis[:, i]).sum()) for i in range(basis.shape[1])]

null_overlap_A = overlap_score(rotA_192, N)
null_overlap_B = overlap_score(rotB_192, N)
par_overlap_A = overlap_score(rotA_192, H.T if H.size else np.zeros((J_seam.shape[0],0),dtype=np.uint8))
par_overlap_B = overlap_score(rotB_192, H.T if H.size else np.zeros((J_seam.shape[0],0),dtype=np.uint8))

print("Right-null overlap with rotA:", null_overlap_A[:10], "... total basis vectors", N.shape[1] if N.size else 0)
print("Right-null overlap with rotB:", null_overlap_B[:10], "... total basis vectors", N.shape[1] if N.size else 0)
print("Left-null overlap with rotA:", par_overlap_A[:10], "... total checks", H.shape[0] if H.size else 0)
print("Left-null overlap with rotB:", par_overlap_B[:10], "... total checks", H.shape[0] if H.size else 0)


## 8. Augmented observables

One of the major structural questions in the recent stack is whether the seam deficit is an artifact of projection rather than a defect of the full die.

So this cell measures rank again after augmenting the seam with transported state tracks.

We test:
- seam only,
- seam + \(d\)-track,
- seam + \(a\)-track,
- seam + \(e\)-track.


In [ ]:

for basis in ["pre_update_xor", "seam_plus_d_track", "seam_plus_a_track", "seam_plus_e_track"]:
    J = build_jacobian_W6(basis)
    rank = gf2_rank(J)
    print(f"{basis:18s} -> shape {J.shape}, rank {rank}/{J.shape[1]}, nullity {J.shape[1]-rank}")


## 9. Hierarchy of absence map

This notebook treats the paper’s hierarchy as a three-resolution object.

### Orbit
Macro-scale open target or final exposed gap.

### Seam
Operational decomposition:
$$
\text{seam}_{\text{full}} = \text{seam}_{\text{xor}} \oplus \text{carry}_{\text{residual}}.
$$

### Jacobian
Foundational blind space:
$$
\operatorname{Im}(J) \subset \mathbb F_2^m
$$

with null directions and parity checks that define the local absence geometry.

The working thesis is:

$$
\boxed{
\text{the same seed manifests as open target at the orbit scale, residual carry at the seam scale, and codimension at the Jacobian scale.}
}
$$


## 10. Universal Seed workbench

The paper centers the seed

$$
B = \Sigma_0(a)\land \operatorname{Maj}(a,b,c)
$$

as the thermodynamic exhaust / folded overlap source.

This cell measures how much of the seam’s carry residual is directly explained by \(B\ll 1\) in the chosen basis.


In [ ]:

obs = observe_orbit([0,0,0,0,0,0], rounds=6)

print("r  B        B<<1     carry_residual   match?")
matches = 0
for row in obs:
    b_shift = ((row.seed_B << 1) & MASK32)
    ok = (b_shift == row.carry_residual)
    if ok:
        matches += 1
    print(f"{row.r:<2d} {row.seed_B:08x} {b_shift:08x} {row.carry_residual:08x}   {ok}")

print(f"\nExact equality count in this basis: {matches}/{len(obs)}")
print("Interpretation:")
print("  • equality means the basis is already aligned with the carry-save reading")
print("  • inequality means the basis still mixes in additional carry/transport structure")


## 11. Parity recovery scaffold

If the seam map is rank-deficient, the inverse problem becomes:

1. choose / measure an observed target seam \(y_{\text{obs}}\),
2. enforce parity:
   $$
   H y = 0
   $$
3. solve:
   $$
   Jx = y
   $$
4. enumerate null-space branches:
   $$
   x = x_p \oplus N\alpha
   $$
5. reintroduce carry residual / seed structure.

This cell provides a scaffold solver for the **linearized** stage. It does not pretend to close the full nonlinear carry problem by itself.


In [ ]:

def gf2_solve_particular(A: np.ndarray, b: np.ndarray):
    """Return one solution x to A x = b over GF(2), or None if inconsistent."""
    A = (A.copy() & 1).astype(np.uint8)
    b = (b.copy() & 1).astype(np.uint8).reshape(-1, 1)
    M = np.concatenate([A, b], axis=1)
    m, n1 = M.shape
    n = n1 - 1
    row = 0
    pivots = []
    for col in range(n):
        pivot = None
        for r in range(row, m):
            if M[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        if pivot != row:
            M[[row, pivot]] = M[[pivot, row]]
        for r in range(m):
            if r != row and M[r, col]:
                M[r] ^= M[row]
        pivots.append(col)
        row += 1
        if row == m:
            break
    # inconsistency check
    for r in range(m):
        if not M[r, :n].any() and M[r, n]:
            return None
    x = np.zeros(n, dtype=np.uint8)
    for i, col in enumerate(pivots):
        x[col] = M[i, n]
    return x

# example: solve for the observable generated by one single-bit toggle
target_x = np.zeros(192, dtype=np.uint8)
target_x[0] = 1  # flip bit 0 of W0
target_y = (J_seam @ target_x) & 1

# parity test
parity_ok = np.all(((H @ target_y) & 1) == 0)
x_part = gf2_solve_particular(J_seam, target_y)

print("target_y passes parity?", parity_ok)
print("particular solution found?", x_part is not None)
if x_part is not None:
    print("Hamming weight of one particular x:", int(x_part.sum()))


## 12. Optional extension points

This notebook is already enough to accompany the paper, but the next exact extensions are obvious:

1. **swap in your canonical Phase 508 seam basis** if it differs from this pre-update basis,
2. **measure the claimed 33–36 bit codimension** under the corrected Phase 1152 mapping,
3. **test whether \(\{2,13,22\}\) and \(\{6,11,25\}\) span the blind subspace** in the corrected basis,
4. **add the orbit target layer explicitly** if you want the 1-bit macro gap as a computed object,
5. **wire in the schedule recovery layer** after choosing the exact carry-free representation you want to treat as the `(64,16)` parity code.

The notebook is built so those can be added without rewriting the core.


## Final collapse

The whole notebook compresses to:

$$
\boxed{
\text{hard die algebra} \to \text{measured seam topology} \to \text{rank / null-space geometry} \to \text{seed / parity / recovery}.
}
$$

That is the computational companion to the paper.
